# 03 · Experimentos de embeddings e recuperação

Compara, em um conjunto de perguntas, a recuperação **densa**, **BM25** e **híbrida (RRF)**,
e verifica se os prefixos `query:`/`passage:` do E5 estão sendo aplicados.

> Requer o índice: `python scripts/build_index.py`.

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import pandas as pd
import matplotlib.pyplot as plt
pd.set_option("display.max_colwidth", 200)

In [ ]:
from voc_rag.config import get_settings
from voc_rag.indexing import SentenceTransformerEmbedder, VectorStore
from voc_rag.retrieval import HybridRetriever, process_query

settings = get_settings()
store = VectorStore.load(settings.vectorstore_dir)
embedder = SentenceTransformerEmbedder(settings.embedding_model)
hybrid = HybridRetriever(store, embedder, use_bm25=True)
dense_only = HybridRetriever(store, embedder, use_bm25=False)
print(store.manifest)

In [ ]:
def show(question, retriever, k=8, auto_filters=False):
    pq = process_query(question, retriever.known_categories, auto_filters=auto_filters)
    cands, stats = retriever.retrieve(pq, k=40)
    rows = [{**store.documents.iloc[c.position][["text", "review_score"]].to_dict(),
             "dense": round(c.dense_score, 3), "bm25_rank": c.bm25_rank, "dense_rank": c.dense_rank}
            for c in cands[:k]]
    return pd.DataFrame(rows)

q = "O produto não foi entregue até hoje"
display(show(q, dense_only))
display(show(q, hybrid))

## Distribuição de similaridade: pergunta no escopo vs. fora do escopo

In [ ]:
import numpy as np
questions = {
    "no escopo": ["Problemas com a entrega", "Produto veio com defeito", "Elogios à qualidade",
                  "Atendimento do vendedor", "Produto diferente do anunciado"],
    "fora do escopo": ["Qual é a capital da Austrália?", "Receita de bolo de chocolate",
                       "Cotação do bitcoin", "Escreva uma função em Python", "Quem ganhou a Copa de 2002?"],
}
rows = []
for group, qs in questions.items():
    for q in qs:
        sims = store.embeddings @ embedder.embed_query(q)
        top = np.sort(sims)[::-1]
        rows.append({"grupo": group, "pergunta": q, "top1": top[0], "top10": top[9], "top50": top[49]})
sim_df = pd.DataFrame(rows).round(3)
sim_df

**Leitura:** o E5 produz similaridades cosseno altas para qualquer texto (anisotropia): as
distribuições de `top1` dos dois grupos se sobrepõem, e nenhum limiar absoluto separa
"no escopo" de "fora do escopo". Por isso o guardrail usa o classificador de escopo por
exemplos (próxima célula) mais sinais lexicais (ver docs/decisoes_tecnicas.md, seção 10).

## Classificador de escopo: margem entre exemplos do domínio e de fora dele

In [ ]:
from voc_rag.generation.scope import ScopeClassifier
scope = ScopeClassifier(embedder)
sim_df["margem_escopo"] = [round(scope.margin(q), 3) for q in sim_df["pergunta"]]
sim_df[["grupo", "pergunta", "top1", "margem_escopo"]]